# Weak-Instrument Frontier | Corrected AR-acceptance recompute (NBE1)

Deviation #11 remediation: regenerates the registered replications (identical streams) and recomputes AR acceptance with the standardized-null vector. Writes `{cid}_arfix.npy` per cell plus `_done` markers; checkpoint zip after every cell.


In [ ]:
import json, os, subprocess, sys, time

REPO_URL = "https://github.com/hugogobato/weakiv-frontier.git"
REPO_REF = "main"
NB_ID = "NBE1"

if not os.path.isdir("weakiv-frontier"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_REF,
                    REPO_URL], check=True)
os.chdir("weakiv-frontier")
GIT_SHA = subprocess.check_output(
    ["git", "rev-parse", "HEAD"]).decode().strip()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e",
                "./Research/spectraliv"], check=True)
sys.path.insert(0, "Research/spectraliv/src")
print("repo sha:", GIT_SHA)


In [ ]:
import json, os, shutil, time, traceback
import numpy as np
from scipy.stats import f as fdist
from spectraliv.dgps import make_single_spike, rho_of_kappa
from spectraliv.preprocess import prepare
from spectraliv.rng import cell_stream

OUT_ROOT = "/content/results"
MASTER_SEED = 20260823
CELLS = [
 {
  "cell_id": "a0.9_k0.5_none_p5_th0.05",
  "reps": 400,
  "q": 1799
 },
 {
  "cell_id": "a0.9_k0.5_none_p5_th0.1",
  "reps": 400,
  "q": 1799
 },
 {
  "cell_id": "a0.9_k0.5_none_p5_th0.18",
  "reps": 400,
  "q": 1799
 },
 {
  "cell_id": "a0.9_k0.5_none_p5_th0.28",
  "reps": 400,
  "q": 1799
 },
 {
  "cell_id": "a0.9_k0.5_none_p5_th0.4",
  "reps": 400,
  "q": 1799
 },
 {
  "cell_id": "a0.9_k0.5_none_p5_th0.55",
  "reps": 400,
  "q": 1799
 },
 {
  "cell_id": "a0.9_k0.5_none_p5_th0.72",
  "reps": 400,
  "q": 1799
 },
 {
  "cell_id": "a0.9_k0.5_none_p5_th0.88",
  "reps": 400,
  "q": 1799
 },
 {
  "cell_id": "a0.9_k2.0_none_p5_th0.05",
  "reps": 400,
  "q": 1799
 },
 {
  "cell_id": "a0.9_k2.0_none_p5_th0.1",
  "reps": 400,
  "q": 1799
 },
 {
  "cell_id": "a0.9_k2.0_none_p5_th0.18",
  "reps": 400,
  "q": 1799
 },
 {
  "cell_id": "a0.9_k2.0_none_p5_th0.28",
  "reps": 400,
  "q": 1799
 },
 {
  "cell_id": "a0.9_k2.0_none_p5_th0.4",
  "reps": 400,
  "q": 1799
 },
 {
  "cell_id": "a0.9_k2.0_none_p5_th0.55",
  "reps": 400,
  "q": 1799
 },
 {
  "cell_id": "a0.9_k2.0_none_p5_th0.72",
  "reps": 400,
  "q": 1799
 },
 {
  "cell_id": "a0.9_k2.0_none_p5_th0.88",
  "reps": 400,
  "q": 1799
 },
 {
  "cell_id": "a0.7_k0.5_none_p5_th0.05",
  "reps": 400,
  "q": 1399
 },
 {
  "cell_id": "a0.7_k0.5_none_p5_th0.1",
  "reps": 400,
  "q": 1399
 },
 {
  "cell_id": "a0.7_k0.5_none_p5_th0.18",
  "reps": 400,
  "q": 1399
 },
 {
  "cell_id": "a0.7_k0.5_none_p5_th0.28",
  "reps": 400,
  "q": 1399
 },
 {
  "cell_id": "a0.7_k0.5_none_p5_th0.4",
  "reps": 400,
  "q": 1399
 },
 {
  "cell_id": "a0.7_k0.5_none_p5_th0.55",
  "reps": 400,
  "q": 1399
 },
 {
  "cell_id": "a0.7_k0.5_none_p5_th0.72",
  "reps": 400,
  "q": 1399
 },
 {
  "cell_id": "a0.7_k0.5_none_p5_th0.88",
  "reps": 400,
  "q": 1399
 },
 {
  "cell_id": "a0.7_k2.0_none_p5_th0.05",
  "reps": 400,
  "q": 1399
 },
 {
  "cell_id": "a0.7_k2.0_none_p5_th0.1",
  "reps": 400,
  "q": 1399
 },
 {
  "cell_id": "a0.7_k2.0_none_p5_th0.18",
  "reps": 400,
  "q": 1399
 },
 {
  "cell_id": "a0.7_k2.0_none_p5_th0.28",
  "reps": 400,
  "q": 1399
 },
 {
  "cell_id": "a0.7_k2.0_none_p5_th0.4",
  "reps": 400,
  "q": 1399
 },
 {
  "cell_id": "a0.7_k2.0_none_p5_th0.55",
  "reps": 400,
  "q": 1399
 }
]

CELLDIR = os.path.join(OUT_ROOT, "phase3_decisive_grid", "cells")
os.makedirs(CELLDIR, exist_ok=True)


def base_of(cid):
    return cid.rsplit("_th", 1)[0]


def parse_cid(cid):
    a_part, rest = cid[1:].split("_k", 1)
    k_part, rest = rest.split("_none_p", 1)
    p_part, th_part = rest.split("_th", 1)
    return float(a_part), float(k_part), int(p_part), float(th_part)


def process(cid, reps):
    a, kap, p, theta = parse_cid(cid)
    n = 1000 if p == 1 else 2000
    q = int(round(a * (n - 1)))
    rho = rho_of_kappa(kap)
    out = np.empty(reps, dtype=bool)
    for b in range(reps):
        rng = np.random.default_rng(
            cell_stream("phase3_decisive_grid", base_of(cid), b,
                        master_seed=MASTER_SEED).integers(1 << 31))
        dgp = make_single_spike(n, q, theta, rho, rng, p=p, beta=0.5)
        xs, zs, yr, resc = prepare(dgp.x, dgp.z, dgp.y, None)
        sy = np.std(yr, ddof=1)
        ys = yr / sy
        chol = np.linalg.cholesky(zs.T @ zs)
        e = ys - xs @ (np.full(p, 0.5) / resc)
        w_e = np.linalg.solve(chol, zs.T @ e)
        num = max(float(w_e @ w_e), 1e-300)
        den = max(float(e @ e) - num, 1e-300)
        stat = (num / q) / (den / (n - q))
        out[b] = stat <= fdist.ppf(0.95, q, n - q)
    return out


def ckpt(tag):
    try:
        zp = "/content/results_NBE_ckpt_%s" % tag
        shutil.make_archive(zp, "zip", OUT_ROOT)
        from google.colab import files
        files.download(zp + ".zip")
        print("[ckpt] downloaded:", zp + ".zip")
    except Exception as e:
        print("(checkpoint download skipped):", e)


summary = []
for item in CELLS:
    cid = item["cell_id"]
    mk = os.path.join(CELLDIR, cid + "_arfix_done.json")
    if os.path.exists(mk):
        print("[skip]", cid)
        continue
    t0 = time.time()
    try:
        arr = process(cid, item["reps"])
        np.save(os.path.join(CELLDIR, cid + "_arfix.npy"), arr)
        with open(mk, "w") as f:
            json.dump({"reps": len(arr), "seed": MASTER_SEED}, f)
        print("[done] %s in %.1fs cov=%.4f" % (cid, time.time() - t0,
                                               arr.mean()))
        summary.append([cid, round(time.time() - t0, 1)])
        ckpt(cid.replace(".", ""))
    except Exception:
        print("[FAIL]", cid)
        traceback.print_exc()

with open(os.path.join(OUT_ROOT, "manifest_ARFIX.json"), "w") as f:
    json.dump({"notebook_id": NB_ID, "git_sha": GIT_SHA,
                "cells": [c["cell_id"] for c in CELLS],
                "timings_s": summary}, f, indent=1)
print(json.dumps(summary, indent=1))


In [ ]:
import shutil, os
os.chdir("/content")
shutil.make_archive("results_NBE1", "zip", "/content/results")
try:
    from google.colab import files
    files.download("results_NBE1.zip")
    print("Downloaded:", "results_NBE1.zip")
except Exception as e:
    print("(Not on Colab / download skipped):", e)
